In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
import os

# --- LOCAL MODULES ---
import sys
sys.path.append('..')
from src.utils import get_device
from src.data import WildfireDataset
from src.models import DynamicMLP, FocalLoss
from src.features import load_and_engineer_features

# --- CONFIGURATION ---
DATA_PATH = "../data/processed/clean_homes.csv"
DEVICE = get_device()

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
## 1. Feature Engineering & Preprocessing

try:
    # Uses the refactored function from src/features.py
    X, y, feature_names = load_and_engineer_features(DATA_PATH)
    print(f"Feature Engineering Complete. Input Shape: {X.shape}")

    # Split & Scale
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

except Exception as e:
    print(f"Data loading failed: {e}. Please ensure processed data exists.")

In [ ]:
## 2. XGBoost Baseline

print("🤖 Training XGBoost Baseline...")
xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    eval_metric='logloss',
    early_stopping_rounds=10,
    n_jobs=-1
)

xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

xgb_acc = accuracy_score(y_test, y_pred_xgb)
xgb_auc = roc_auc_score(y_test, y_prob_xgb)

print(f"🏆 XGBoost Results -- Accuracy: {xgb_acc:.4f} | AUC: {xgb_auc:.4f}")

In [ ]:
## 3. Deep Learning: MLP with Focal Loss

train_loader = DataLoader(WildfireDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(WildfireDataset(X_val, y_val), batch_size=64)
test_loader = DataLoader(WildfireDataset(X_test, y_test), batch_size=64)

def train_mlp(config):
    model = DynamicMLP(X_train.shape[1], config['layers'], config['dropout']).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = FocalLoss(alpha=0.25, gamma=2)
    
    best_auc = 0
    patience = 5
    counter = 0
    
    for epoch in range(50):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            out = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            optimizer.step()
            
        # Validation
        model.eval()
        all_preds = []
        all_targets = []
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b = X_b.to(DEVICE)
                out = torch.sigmoid(model(X_b))
                all_preds.extend(out.cpu().numpy())
                all_targets.extend(y_b.numpy())
        
        val_auc = roc_auc_score(all_targets, all_preds)
        if val_auc > best_auc:
            best_auc = val_auc
            counter = 0
            best_state = model.state_dict()
        else:
            counter += 1
            if counter >= patience: break
            
    return best_auc, best_state

In [ ]:
# Search configs
configs = [
    {'layers': [64, 32], 'dropout': 0.2},
    {'layers': [128, 64], 'dropout': 0.3},
    {'layers': [256, 128, 64], 'dropout': 0.4}
]

print("🧪 Searching MLP Architectures...")
best_mlp_auc = 0
best_mlp_model = None
best_conf = None

for conf in configs:
    auc, state = train_mlp(conf)
    print(f"Config {conf['layers']} -> AUC: {auc:.4f}")
    if auc > best_mlp_auc:
        best_mlp_auc = auc
        best_conf = conf
        best_mlp_model = DynamicMLP(X_train.shape[1], conf['layers'], conf['dropout']).to(DEVICE)
        best_mlp_model.load_state_dict(state)

print(f"\n🏆 Best MLP Config: {best_conf['layers']} (AUC: {best_mlp_auc:.4f})")

In [ ]:
## 4. Final Comparison & Selection

# Evaluate Best MLP on Test Set
best_mlp_model.eval()
mlp_probs = []
mlp_targets = []
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(DEVICE)
        out = torch.sigmoid(best_mlp_model(X_b))
        mlp_probs.extend(out.cpu().numpy())
        mlp_targets.extend(y_b.numpy())

mlp_test_auc = roc_auc_score(mlp_targets, mlp_probs)
mlp_test_acc = accuracy_score(mlp_targets, (np.array(mlp_probs) > 0.5).astype(int))

print("="*40)
print("📊 FINAL TEST SET RESULTS")
print(f"XGBoost  -> Accuracy: {xgb_acc:.4f} | AUC: {xgb_auc:.4f}")
print(f"Best MLP -> Accuracy: {mlp_test_acc:.4f} | AUC: {mlp_test_auc:.4f}")
print("="*40)

if xgb_auc > mlp_test_auc:
    print("✅ XGBoost wins! Saving XGBoost model...")
    # xgb.save_model("../models/best_model.json")
else:
    print("✅ MLP wins! Saving PyTorch model...")
    torch.save(best_mlp_model.state_dict(), "../models/best_model.pth")

# Comparison Plot
labels = ['XGBoost', 'MLP (Focal Loss)']
aucs = [xgb_auc, mlp_test_auc]

plt.figure(figsize=(6, 5))
plt.bar(labels, aucs, color=['skyblue', 'salmon'])
plt.ylim(0, 1.0)
plt.title('Final AUC Comparison')
plt.ylabel('ROC-AUC Score')
for i, v in enumerate(aucs):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center')
plt.show()